In [17]:
# ============================================================
# BUILD DIM_RACES — LOCAL EXECUTION
# ============================================================

import os
from pyspark.sql import functions as F
import nbformat
from nbconvert import PythonExporter

In [18]:
# -----------------------------------------
# 1. Load environment + helpers
# -----------------------------------------
def run_notebook(path):
    with open(path, "r") as f:
        nb = nbformat.read(f, as_version=4)
    source, _ = PythonExporter().from_notebook_node(nb)
    exec(source, globals())

run_notebook("/Users/manoharazalki/F1-Analytics/notebooks/00-common/01_environment_config.ipynb")
run_notebook("/Users/manoharazalki/F1-Analytics/notebooks/00-common/04_gold_helpers.ipynb")

===== F1 Analytics Environment Configuration =====
BASE_PATH:        /Users/manoharazalki/F1-Analytics/data
LANDING_PATH:     /Users/manoharazalki/F1-Analytics/data/landing
BRONZE_PATH:      /Users/manoharazalki/F1-Analytics/data/bronze
SILVER_PATH:      /Users/manoharazalki/F1-Analytics/data/silver
GOLD_PATH:        /Users/manoharazalki/F1-Analytics/data/gold
INCREMENTAL_PATH: /Users/manoharazalki/F1-Analytics/data/incremental
CONTROL_PATH:     /Users/manoharazalki/F1-Analytics/data/control


In [19]:
# -----------------------------------------
# 2. Receive p_batch_id (local + Databricks)
# -----------------------------------------

# Case 1: Papermill parameter (local pipeline)
if "p_batch_id" in globals() and p_batch_id not in [None, ""]:
    pass

# Case 2: Databricks widget
elif "dbutils" in globals():
    p_batch_id = dbutils.widgets.get("p_batch_id")

# No auto-detection for Gold layer (Silver is not partitioned)
else:
    raise Exception("❌ p_batch_id not provided to Gold notebook")

print("Using p_batch_id:", p_batch_id)


Exception: ❌ p_batch_id not provided to Gold notebook

In [ ]:
# -----------------------------------------
# 3. Define Silver + Gold paths
# -----------------------------------------
silver_races = f"{SILVER_PATH}/races/data.parquet"
silver_circuits = f"{SILVER_PATH}/circuits/data.parquet"

gold_path = f"{GOLD_PATH}/dim_races"
os.makedirs(gold_path, exist_ok=True)

In [ ]:
# -----------------------------------------
# 4. Read Silver (ONLY this batch)
# -----------------------------------------
races_df = (
    spark.read.parquet(silver_races_path)
    .filter(F.col("batch_id") == p_batch_id)
)

circuits_df = (
    spark.read.parquet(silver_circuits_path)
    .filter(F.col("batch_id") == p_batch_id)
)

print("✔ Silver races rows:", races_df.count())
print("✔ Silver circuits rows:", circuits_df.count())

if races_df.count() == 0:
    raise Exception("❌ Silver races has 0 rows for this batch_id")

if circuits_df.count() == 0:
    raise Exception("❌ Silver circuits has 0 rows for this batch_id")

NameError: name 'silver_races_path' is not defined

In [ ]:
# -----------------------------------------
# 5. Join + Select required columns
# -----------------------------------------
dim_races_df = (
    races_df.alias("r")
        .join(
            circuits_df.alias("c"),
            F.col("r.circuit_id") == F.col("c.circuit_id"),
            "inner"
        )
        .select(
            F.col("r.year").alias("season"),
            F.col("r.round"),
            F.col("r.race_name"),
            F.col("r.race_date"),
            F.col("c.circuit_name"),
            F.col("c.locality"),
            F.col("c.country_name")
        )
)

print("✔ dim_races_df rows:", dim_races_df.count())

NameError: name 'races_df' is not defined

In [ ]:
# -----------------------------------------
# 6. Write to Gold
# -----------------------------------------
write_to_gold(
    input_df=dim_races_df,
    target_path=f"{gold_path}/data.parquet",
    merge_keys=["season", "round"]
)

print("✔ dim_races written:", f"{gold_path}/data.parquet")

NameError: name 'dim_races_df' is not defined

In [ ]:
# -----------------------------------------
# 7. Validate
# -----------------------------------------
spark.read.parquet(f"{gold_path}/data.parquet").show(5, truncate=False)